# V-ChartText – Chạy lại các output bị cắt ở 768 token (max_new_tokens = 2048)

Decode greedy là tất định: với cùng ảnh, prompt, mô hình và phiên bản thư viện, 768 token đầu giữ nguyên, chỉ phần sau được sinh thêm.
Vì vậy chỉ cần chạy lại các id đã chạm 768 token; kết quả tương đương chạy lại toàn bộ với `max_new_tokens = 2048`. Cell 7 kiểm tra điều này.

Mỗi lần *Save Version* chạy **một** mô hình:
- Lần A: `FAMILY = 'qwen'`, Input = dataset ảnh + **Notebook Output của lần chạy baseline Vintern** (có đủ 4 file `.jsonl`).
- Lần B: `FAMILY = 'vintern'`, Input = dataset ảnh + **Notebook Output của lần A** (bỏ input cũ đi).

Chỉ để **một** Notebook Output trong Input, nếu không cell 2 có thể chép nhầm file cũ.
Phiên bản transformers được ghim đúng như lần chạy baseline (Qwen 5.20.2, Vintern 4.49.0).

In [ ]:
FAMILY  = 'qwen'            # 'qwen' cho lần A, 'vintern' cho lần B
GH_USER = 'rangha26'
BRANCH  = 'dev'
REPO    = '/tmp/V-ChartText'
OUT     = '/kaggle/working/outputs'
QWEN    = 'Qwen/Qwen3-VL-2B-Instruct'
VINTERN = '5CD-AI/Vintern-1B-v3_5'
NEW_MAX = 2048

import glob, os, json, unicodedata, shutil
nfc = lambda s: unicodedata.normalize('NFC', s).lower()
RAW = next(d for d in sorted(glob.glob('/kaggle/input/**/', recursive=True))
           if any('cột' in nfc(x) for x in os.listdir(d)))
print('FAMILY =', FAMILY, '| RAW =', RAW)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Chép 4 file kết quả từ Notebook Output đã Add Input
os.makedirs(OUT, exist_ok=True)
src = sorted(glob.glob('/kaggle/input/**/outputs/*.jsonl', recursive=True))
assert len(src) == 4, f'Cần đúng 4 file .jsonl trong Input, đang thấy {len(src)}: {src}'
for f in src:
    shutil.copy(f, os.path.join(OUT, os.path.basename(f))); print('chép', f)
# bản sao lưu 768 token của lần A (nếu có) để cell 7 so sánh ở lần B
for f in glob.glob('/kaggle/input/**/outputs_v768/*.jsonl', recursive=True):
    os.makedirs('/kaggle/working/outputs_v768', exist_ok=True)
    shutil.copy(f, '/kaggle/working/outputs_v768/'); print('chép sao lưu', f)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GH_TOKEN')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://{token}@github.com/{GH_USER}/V-ChartText.git {REPO}
%cd {REPO}
!git clean -fq -- data/labels data/manifest.csv
!git pull -q origin {BRANCH}
!git log --oneline -1
if not os.path.islink('outputs'):
    !rm -rf outputs && ln -s {OUT} outputs
!wc -l outputs/*.jsonl

In [ ]:
# Ghim đúng phiên bản của lần chạy baseline
if FAMILY == 'qwen':
    !pip install -q "transformers==5.20.2" accelerate
else:
    !pip install -q "transformers==4.49.0" timm einops sentencepiece
!python -c "import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)"
!python src/prepare_dataset.py --raw "{RAW}" | head -3

In [ ]:
# Tìm output chạm 768 token, sao lưu bản gốc sang /kaggle/working/outputs_v768 rồi xoá các dòng đó
BK = '/kaggle/working/outputs_v768'
if FAMILY == 'qwen':
    !python src/find_truncated.py outputs/Qwen3-VL-2B-Instruct__*.jsonl --backup {BK} --drop
else:
    !python src/find_truncated.py outputs/Vintern-1B-v3_5__*.jsonl --tokenizer {VINTERN} --backup {BK} --drop
RERUN = json.load(open('rerun_ids.json', encoding='utf-8'))
ids = {('extract' if '__extract__' in f else 'describe'): ','.join(v) for f, v in RERUN.items()}
print({k: len(v.split(',')) if v else 0 for k, v in ids.items()})

In [ ]:
# Chạy lại: extract trên GPU 0, describe trên GPU 1, song song
if FAMILY == 'qwen':
    CMD = f'python src/infer_qwen.py --model {QWEN} --max_new_tokens {NEW_MAX}'
else:
    CMD = f'python src/infer_vintern.py --repetition_penalty 1.0 --max_new_tokens {NEW_MAX}'
ex, de = ids.get('extract', ''), ids.get('describe', '')
jobs = []
if ex: jobs.append(f'CUDA_VISIBLE_DEVICES=0 {CMD} --task extract --ids "{ex}" > /kaggle/working/rerun_{FAMILY}_extract.txt 2>&1')
if de: jobs.append(f'CUDA_VISIBLE_DEVICES=1 {CMD} --task describe --ids "{de}" > /kaggle/working/rerun_{FAMILY}_describe.txt 2>&1')
if jobs:
    script = ' & '.join(jobs) + ' & wait'
    !bash -c '{script}'
    !tail -n 4 /kaggle/working/rerun_{FAMILY}_*.txt
else:
    print('Không có output nào bị cắt, không cần chạy lại.')
!wc -l outputs/*.jsonl

In [ ]:
# Kiểm tra: đủ 215 dòng, 768 token đầu trùng bản cũ (tính tất định), còn bài nào chạm 2048 không, JSON hợp lệ
import re
def load(f): return {json.loads(l)['id']: json.loads(l) for l in open(f, encoding='utf-8') if l.strip()}
def json_ok(t):
    t = re.sub(r'```(?:json)?', '', t); m = re.search(r'\{.*\}', t, re.S)
    try: json.loads(m.group(0)); return True
    except Exception: return False
for f, rerun in RERUN.items():
    if not rerun: continue
    new, old = load(f), load(os.path.join(BK, os.path.basename(f)))
    same = [i for i in rerun if new[i]['output'].startswith(old[i]['output'].rstrip()[:-20])]
    still = [i for i in rerun if (new[i].get('n_new_tokens') or 0) >= NEW_MAX - 8]
    print(f'{os.path.basename(f)}: {len(new)} dòng | chạy lại {len(rerun)} | phần đầu trùng bản cũ {len(same)}/{len(rerun)}'
          f' | vẫn chạm {NEW_MAX}: {len(still)} {still}')
    if '__extract__' in f:
        print('   JSON hợp lệ trong số chạy lại:', sum(json_ok(new[i]['output']) for i in rerun), '/', len(rerun))
    diff = [i for i in rerun if i not in same]
    if diff: print('   KHÁC phần đầu (xem lại phiên bản thư viện):', diff)

In [ ]:
# Chấm lại (chỉ ảnh đã có nhãn) và lưu kết quả
for f in sorted(glob.glob('outputs/*__extract__*.jsonl')):
    !python eval/score_extract.py "{f}"
for f in sorted(glob.glob('outputs/*__describe__*.jsonl')):
    !python eval/check_describe.py "{f}"
!mkdir -p /kaggle/working/results && cp -r eval/results/* /kaggle/working/results/ 2>/dev/null
!cp rerun_ids.json /kaggle/working/rerun_ids_{FAMILY}.json; ls /kaggle/working /kaggle/working/outputs